# 🛍️ Retail Promotions Data Analysis
### Codebasics Assignment — All 10 Questions

**Key Metrics:**
- **IR%** = ((Revenue After Promo - Revenue Before Promo) / Revenue Before Promo) × 100
- **ISU%** = ((Qty Sold After Promo - Qty Sold Before Promo) / Qty Sold Before Promo) × 100

## 📦 Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ─── Load all 4 datasets ───────────────────────────────────────────────────
campaigns = pd.read_csv('dim_campaigns.csv')
products  = pd.read_csv('dim_products.csv')
stores    = pd.read_csv('dim_stores.csv')
events    = pd.read_csv('fact_events.csv')

print('✅ Datasets loaded successfully!')
print(f'   dim_campaigns : {campaigns.shape}')
print(f'   dim_products  : {products.shape}')
print(f'   dim_stores    : {stores.shape}')
print(f'   fact_events   : {events.shape}')

In [ ]:
# Quick preview of each table
print('=== dim_campaigns ===')
display(campaigns.head(3))

print('\n=== dim_products ===')
display(products.head(3))

print('\n=== dim_stores ===')
display(stores.head(3))

print('\n=== fact_events ===')
display(events.head(3))

---
## ❓ Q1 — Remove Duplicate Rows in Events Data
**Task:** Remove duplicates based on `store_id`, `campaign_id`, and `product_code`. How many duplicate rows were removed?

In [ ]:
rows_before = len(events)

events_clean = events.drop_duplicates(subset=['store_id', 'campaign_id', 'product_code'])

rows_after   = len(events_clean)
duplicates_removed = rows_before - rows_after

print(f'Rows before deduplication : {rows_before}')
print(f'Rows after  deduplication : {rows_after}')
print(f'✅ Duplicate rows removed  : {duplicates_removed}')

---
## ❓ Q2 — Cities with More Than 5 Stores

In [ ]:
city_store_count = stores.groupby('city')['store_id'].count().reset_index()
city_store_count.columns = ['city', 'store_count']

cities_more_than_5 = city_store_count[city_store_count['store_count'] > 5]

print(f'✅ Number of cities with more than 5 stores: {len(cities_more_than_5)}')
print()
display(cities_more_than_5.sort_values('store_count', ascending=False).reset_index(drop=True))

---
## ❓ Q3 — Fill Missing Values in `quantity_sold(before_promo)` Using Median
**Task:** Impute missing values using the median. How many were filled? What is the median?

In [ ]:
col = 'quantity_sold(before_promo)'

missing_count = events_clean[col].isnull().sum()
median_value  = events_clean[col].median()

events_clean[col] = events_clean[col].fillna(median_value)

print(f'✅ Missing values filled : {missing_count}')
print(f'✅ Median used           : {median_value}')

---
## ❓ Q4 — Product Category with the Lowest Base Price (Before Promo)

In [ ]:
# Merge events with products to get category info
events_prod = events_clean.merge(products, on='product_code', how='left')

avg_base_price_by_category = (
    events_prod
    .groupby('category')['base_price(before_promo)']
    .mean()
    .reset_index()
    .sort_values('base_price(before_promo)')
)

lowest_category = avg_base_price_by_category.iloc[0]

print(f'✅ Category with LOWEST avg base price: {lowest_category["category"]}')
print(f'   Average base price: ₹{lowest_category["base_price(before_promo)"]:.2f}')
print()
display(avg_base_price_by_category.reset_index(drop=True))

---
## ❓ Q5 — Total Quantity Sold After Promo for BOGOF During Diwali Campaign

In [ ]:
# Merge with campaigns
events_full = events_clean.merge(campaigns, on='campaign_id', how='left')

# BOGOF: customer pays for 1, gets 2 — so actual units = quantity_sold(after_promo) * 2
# Note: Some interpretations keep it as-is; adjust the multiplier below if needed
bogof_diwali = events_full[
    (events_full['campaign_name'].str.lower() == 'diwali') &
    (events_full['promo_type'].str.upper() == 'BOGOF')
]

total_qty_bogof_diwali = bogof_diwali['quantity_sold(after_promo)'].sum()

# For BOGOF, actual units dispensed = qty * 2
actual_units_bogof = total_qty_bogof_diwali * 2

print(f'✅ Total quantity_sold(after_promo) for BOGOF / Diwali: {total_qty_bogof_diwali}')
print(f'   (Actual units incl. free items @ BOGOF)           : {actual_units_bogof}')

---
## ❓ Q6 — Store with Highest Quantity Sold After Promo During Diwali

In [ ]:
diwali_events = events_full[events_full['campaign_name'].str.lower() == 'diwali']

store_qty_diwali = (
    diwali_events
    .groupby('store_id')['quantity_sold(after_promo)']
    .sum()
    .reset_index()
    .sort_values('quantity_sold(after_promo)', ascending=False)
)

# Merge city info
store_qty_diwali = store_qty_diwali.merge(stores, on='store_id', how='left')

top_store = store_qty_diwali.iloc[0]

print(f'✅ Store with HIGHEST qty sold after promo (Diwali):')
print(f'   Store ID : {top_store["store_id"]}')
print(f'   City     : {top_store["city"]}')
print(f'   Qty Sold : {top_store["quantity_sold(after_promo)"]}')
print()
display(store_qty_diwali.head(5).reset_index(drop=True))

---
## ❓ Q7 — Compare Sankranti vs Diwali: Which Campaign Saw Greater Sales Increase?

In [ ]:
campaign_comparison = (
    events_full[
        events_full['campaign_name'].str.lower().isin(['diwali', 'sankranti'])
    ]
    .groupby('campaign_name')
    .agg(
        total_qty_before=('quantity_sold(before_promo)', 'sum'),
        total_qty_after =('quantity_sold(after_promo)',  'sum')
    )
    .reset_index()
)

campaign_comparison['increase']   = campaign_comparison['total_qty_after'] - campaign_comparison['total_qty_before']
campaign_comparison['increase_%'] = ((campaign_comparison['increase'] / campaign_comparison['total_qty_before']) * 100).round(2)

display(campaign_comparison)

winner = campaign_comparison.loc[campaign_comparison['increase'].idxmax(), 'campaign_name']
print(f'\n✅ Campaign with GREATER increase in sales: {winner}')

---
## ❓ Q8 — Product with Highest IR% During Sankranti Campaign

**Formula:**
```
Revenue Before = base_price(before_promo) × quantity_sold(before_promo)
Revenue After  = base_price(after_promo)  × quantity_sold(after_promo)
IR% = ((Revenue After - Revenue Before) / Revenue Before) × 100
```

In [ ]:
sankranti = events_full[
    events_full['campaign_name'].str.lower() == 'sankranti'
].copy()

sankranti['revenue_before'] = sankranti['base_price(before_promo)'] * sankranti['quantity_sold(before_promo)']
sankranti['revenue_after']  = sankranti['base_price(after_promo)']  * sankranti['quantity_sold(after_promo)']

product_ir = (
    sankranti
    .groupby('product_code')
    .agg(
        rev_before=('revenue_before', 'sum'),
        rev_after =('revenue_after',  'sum')
    )
    .reset_index()
)

product_ir['IR%'] = ((product_ir['rev_after'] - product_ir['rev_before']) / product_ir['rev_before'] * 100).round(2)

# Merge product names
product_ir = product_ir.merge(products[['product_code', 'product_name', 'category']], on='product_code', how='left')
product_ir = product_ir.sort_values('IR%', ascending=False)

top_product = product_ir.iloc[0]

print(f'✅ Product with HIGHEST IR% during Sankranti:')
print(f'   Product  : {top_product["product_name"]}')
print(f'   Category : {top_product["category"]}')
print(f'   IR%      : {top_product["IR%"]}%')
print()
display(product_ir.head(5).reset_index(drop=True))

---
## ❓ Q9 — Store in Visakhapatnam with Lowest ISU% During Diwali

**Formula:**
```
ISU% = ((quantity_sold(after_promo) - quantity_sold(before_promo)) / quantity_sold(before_promo)) × 100
```

In [ ]:
# Merge stores into events_full
events_full_stores = events_full.merge(stores, on='store_id', how='left')

vizag_diwali = events_full_stores[
    (events_full_stores['campaign_name'].str.lower() == 'diwali') &
    (events_full_stores['city'].str.lower() == 'visakhapatnam')
].copy()

store_isu = (
    vizag_diwali
    .groupby('store_id')
    .agg(
        qty_before=('quantity_sold(before_promo)', 'sum'),
        qty_after =('quantity_sold(after_promo)',  'sum')
    )
    .reset_index()
)

store_isu['ISU%'] = ((store_isu['qty_after'] - store_isu['qty_before']) / store_isu['qty_before'] * 100).round(2)
store_isu = store_isu.sort_values('ISU%')

lowest_store = store_isu.iloc[0]

print(f'✅ Store in Visakhapatnam with LOWEST ISU% (Diwali):')
print(f'   Store ID : {lowest_store["store_id"]}')
print(f'   ISU%     : {lowest_store["ISU%"]}%')
print()
display(store_isu.reset_index(drop=True))

---
## ❓ Q10 — Promo Type with BOTH Negative IR% AND Negative ISU% During Sankranti

In [ ]:
sankranti2 = events_full[
    events_full['campaign_name'].str.lower() == 'sankranti'
].copy()

sankranti2['revenue_before'] = sankranti2['base_price(before_promo)'] * sankranti2['quantity_sold(before_promo)']
sankranti2['revenue_after']  = sankranti2['base_price(after_promo)']  * sankranti2['quantity_sold(after_promo)']

promo_metrics = (
    sankranti2
    .groupby('promo_type')
    .agg(
        qty_before =('quantity_sold(before_promo)', 'sum'),
        qty_after  =('quantity_sold(after_promo)',  'sum'),
        rev_before =('revenue_before',              'sum'),
        rev_after  =('revenue_after',               'sum')
    )
    .reset_index()
)

promo_metrics['IR%']  = ((promo_metrics['rev_after']  - promo_metrics['rev_before'])  / promo_metrics['rev_before']  * 100).round(2)
promo_metrics['ISU%'] = ((promo_metrics['qty_after']  - promo_metrics['qty_before'])  / promo_metrics['qty_before']  * 100).round(2)

display(promo_metrics[['promo_type', 'IR%', 'ISU%']])

negative_both = promo_metrics[
    (promo_metrics['IR%'] < 0) & (promo_metrics['ISU%'] < 0)
]

print(f'\n✅ Promo type(s) with BOTH negative IR% and ISU% (Sankranti):')
if len(negative_both) > 0:
    for _, row in negative_both.iterrows():
        print(f'   {row["promo_type"]} → IR%: {row["IR%"]}%  |  ISU%: {row["ISU%"]}%')
else:
    print('   None found.')

---
## 📊 Summary of All Answers

In [ ]:
print('=' * 60)
print('          SUMMARY OF ALL ANSWERS')
print('=' * 60)
print(f'Q1  Duplicate rows removed          : {duplicates_removed}')
print(f'Q2  Cities with > 5 stores          : {len(cities_more_than_5)}')
print(f'Q3  Missing values filled           : {missing_count}  |  Median: {median_value}')
print(f'Q4  Category with lowest base price : {lowest_category["category"]}')
print(f'Q5  BOGOF Diwali qty (after promo)  : {total_qty_bogof_diwali}')
print(f'Q6  Top store (Diwali)              : {top_store["store_id"]} — {top_store["city"]}')
print(f'Q7  Campaign with greater increase  : {winner}')
print(f'Q8  Highest IR% product (Sankranti) : {top_product["product_name"]} ({top_product["IR%"]}%)')
print(f'Q9  Lowest ISU% store in Vizag      : {lowest_store["store_id"]} ({lowest_store["ISU%"]}%)')
print(f'Q10 Promo with -ve IR% & -ve ISU%   : See output above')
print('=' * 60)